# GPU Experiment: Uncertainty Quantification Methods Comparison

**Dataset:** FB15k-237 (14,541 entities, 237 relations)

**Models:**
1. DistMult (baseline)
2. DistMult + MC Dropout
3. GGPN (Graph Gaussian Process Network)
4. GP-KGE (Our method)

**Metrics:**
- Link Prediction: MRR, Hits@1, Hits@3, Hits@10
- Calibration: ECE (Expected Calibration Error), Brier Score
- OOD Detection: AUROC

---
## Setup (Run First)
---

In [ ]:
# Clone and setup
!git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior 2>/dev/null || (cd /content/kg-bayesian-prior && git pull)
%cd /content/kg-bayesian-prior
!pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn matplotlib

import sys
sys.path.insert(0, '/content/kg-bayesian-prior')

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Imports
import json
import gc
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

from src.data import load_fb15k237
from src.models import DistMult, GPKGE
from src.models.ggpn import GGPN
from src.models.uncertain_kge import MCDropoutKGE
from src.utils.training import set_seed, NegativeSampler
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

set_seed(42)
device = "cuda"
train_data, valid_data, test_data = load_fb15k237()
print(f"Loaded: {len(train_data):,} train, {len(test_data):,} test triples")

results = {}

In [ ]:
# Helper functions
def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

def evaluate_model(model, test_data, train_data, device, model_name="Model"):
    """Evaluate model on all metrics."""
    model.eval()

    # Link Prediction (MRR)
    sample = test_data.triples[np.random.choice(len(test_data), 2000, replace=False)]
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 200), desc="MRR", leave=False):
            batch = sample[i:i+200]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r) if hasattr(model, 'score_tails') else None
            if scores is None:
                all_t = torch.arange(test_data.num_entities, device=device)
                scores = torch.stack([model(h[i].expand(test_data.num_entities), r[i].expand(test_data.num_entities), all_t) for i in range(len(h))])
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    hits1 = (ranks <= 1).float().mean().item()
    hits3 = (ranks <= 3).float().mean().item()
    hits10 = (ranks <= 10).float().mean().item()

    # Calibration (ECE)
    pos = test_data.triples[np.random.choice(len(test_data), 1000, replace=False)]
    neg = np.array([[h, r, np.random.randint(test_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    with torch.no_grad():
        h, r, t = [torch.tensor(all_t[:,j], device=device) for j in range(3)]
        if hasattr(model, 'base_model'):
            scores = model.base_model.score_triple(h, r, t)
        elif hasattr(model, 'score_triple'):
            scores = model.score_triple(h, r, t)
        else:
            scores = model(h, r, t)
        conf = torch.sigmoid(scores).cpu().numpy()

    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)

    # OOD Detection (AUROC)
    id_t = test_data.triples[np.random.choice(len(test_data), 1000, replace=False)]
    ood_t = create_ood_dataset(train_data, test_data, "random", 1000)

    def get_unc(triples):
        with torch.no_grad():
            h, r, t = [torch.tensor(triples[:,j], device=device) for j in range(3)]
            if hasattr(model, 'predict_with_uncertainty'):
                pred = model.predict_with_uncertainty(h, r, t)
                return pred.get('total', pred.get('epistemic', torch.zeros(len(h)))).cpu().numpy() if isinstance(pred, dict) else pred[1].cpu().numpy()
            elif hasattr(model, 'predict_with_mc_samples'):
                _, var = model.predict_with_mc_samples(h, r, t, num_samples=10)
                return var.cpu().numpy()
            else:
                if hasattr(model, 'score_triple'):
                    scores = model.score_triple(h, r, t)
                else:
                    scores = model(h, r, t)
                probs = torch.sigmoid(scores)
                return (-probs * torch.log(probs + 1e-10) - (1-probs) * torch.log(1-probs + 1e-10)).cpu().numpy()

    auroc = compute_auroc(get_unc(id_t), get_unc(ood_t))

    result = {"mrr": mrr, "hits@1": hits1, "hits@3": hits3, "hits@10": hits10, "ece": ece, "brier": brier, "auroc": auroc}
    print(f"{model_name}: MRR={mrr:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return result

print("Ready!")

---
## Model 1: DistMult (Baseline)
---

In [ ]:
clear_memory()
print("=" * 50)
print("MODEL 1/4: DistMult")
print("=" * 50)

distmult = DistMult(train_data.num_entities, train_data.num_relations, embedding_dim=200).to(device)
optimizer = torch.optim.Adam(distmult.parameters(), lr=0.001)
neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
neg_sampler.set_true_triples(train_data.triples)

for epoch in (pbar := tqdm(range(50), desc="DistMult")):
    distmult.train()
    total_loss, n = 0, 0
    for start in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[start:start+1024], device=device)
        neg = neg_sampler(pos).to(device)
        optimizer.zero_grad()
        pos_s = distmult(pos[:,0], pos[:,1], pos[:,2])
        neg_s = distmult(neg[:,0], neg[:,1], neg[:,2]).view(len(pos), -1).mean(1)
        loss = F.relu(1.0 - pos_s + neg_s).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{total_loss/n:.4f}")

results["DistMult"] = evaluate_model(distmult, test_data, train_data, device, "DistMult")
del distmult
clear_memory()

---
## Model 2: DistMult + MC Dropout
---

In [ ]:
clear_memory()
print("=" * 50)
print("MODEL 2/4: DistMult + MC Dropout")
print("=" * 50)

base = DistMult(train_data.num_entities, train_data.num_relations, embedding_dim=200, dropout=0.3)
mc_dropout = MCDropoutKGE(base, num_samples=20).to(device)
optimizer = torch.optim.Adam(mc_dropout.parameters(), lr=0.001)
neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
neg_sampler.set_true_triples(train_data.triples)

for epoch in (pbar := tqdm(range(50), desc="MCDropout")):
    mc_dropout.train()
    total_loss, n = 0, 0
    for start in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[start:start+1024], device=device)
        neg = neg_sampler(pos).to(device)
        optimizer.zero_grad()
        pos_s = mc_dropout.base_model(pos[:,0], pos[:,1], pos[:,2])
        neg_s = mc_dropout.base_model(neg[:,0], neg[:,1], neg[:,2]).view(len(pos), -1).mean(1)
        loss = F.relu(1.0 - pos_s + neg_s).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{total_loss/n:.4f}")

results["DistMult+MCDropout"] = evaluate_model(mc_dropout, test_data, train_data, device, "MCDropout")
del mc_dropout, base
clear_memory()

---
## Model 3: GGPN
---

In [ ]:
clear_memory()
print("=" * 50)
print("MODEL 3/4: GGPN")
print("=" * 50)
print("Using reduced params for GPU memory")

ggpn = GGPN(train_data.num_entities, train_data.num_relations * 2, embedding_dim=50, hidden_dim=50, num_layers=1, num_rff=20).to(device)
ggpn.set_graph(train_data)
optimizer = torch.optim.Adam(ggpn.parameters(), lr=0.001)
neg_sampler = NegativeSampler(train_data.num_entities, num_negatives=10)
neg_sampler.set_true_triples(train_data.triples)

for epoch in (pbar := tqdm(range(50), desc="GGPN")):
    ggpn.train()
    total_loss, n = 0, 0
    for start in range(0, len(train_data), 512):
        pos = torch.tensor(train_data.triples[start:start+512], device=device)
        neg = neg_sampler(pos).to(device)
        optimizer.zero_grad()
        loss = ggpn.loss(pos, neg)
        loss = loss['total'] if isinstance(loss, dict) else loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ggpn.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{total_loss/n:.4f}")

results["GGPN"] = evaluate_model(ggpn, test_data, train_data, device, "GGPN")
del ggpn
clear_memory()

---
## Save Partial Results (Before GP-KGE)
---

In [ ]:
# Save partial results
with open('/content/partial_results.json', 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in results.items()}, f, indent=2)

print("Saved partial results:")
for name, r in results.items():
    print(f"  {name}: MRR={r['mrr']:.4f}, ECE={r['ece']:.4f}")

---
## Model 4: GP-KGE (Our Method)

**A100 GPU 권장** - Runtime > Change runtime type > A100 선택

---

In [ ]:
# GP-KGE - Run this after changing to A100 GPU
# If runtime was restarted, run this cell first:

# === SETUP (run if runtime was restarted) ===
import sys
sys.path.insert(0, '/content/kg-bayesian-prior')

import json, gc, torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

set_seed(42)
device = "cuda"
train_data, valid_data, test_data = load_fb15k237()

# Load previous results
try:
    with open('/content/partial_results.json') as f:
        results = json.load(f)
    print(f"Loaded previous: {list(results.keys())}")
except:
    results = {}
    print("Starting fresh")

print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# === GP-KGE Training ===
print("=" * 50)
print("MODEL 4/4: GP-KGE (Ours)")
print("=" * 50)

gc.collect()
torch.cuda.empty_cache()

gpkge = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=100,
    kernel_type="rbf",
    num_inducing=200,
).to(device)

optimizer = torch.optim.Adam(gpkge.parameters(), lr=0.001)

for epoch in (pbar := tqdm(range(50), desc="GP-KGE")):
    gpkge.train()
    total_loss, n = 0, 0

    for start in range(0, len(train_data), 2048):
        pos = torch.tensor(train_data.triples[start:start+2048], device=device)
        neg = pos.clone()
        neg[:, 2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)

        optimizer.zero_grad()

        # Direct BCE loss (skip KL for speed)
        pos_scores = gpkge.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        neg_scores = gpkge.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)

        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        loss = F.binary_cross_entropy_with_logits(scores, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpkge.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n += 1

    pbar.set_postfix(loss=f"{total_loss/n:.4f}")

print("Training complete!")

In [ ]:
# === GP-KGE Evaluation ===
print("Evaluating GP-KGE...")
gpkge.eval()

# MRR
sample = test_data.triples[np.random.choice(len(test_data), 2000, replace=False)]
ranks = []
with torch.no_grad():
    for i in tqdm(range(0, len(sample), 200), desc="MRR"):
        batch = sample[i:i+200]
        h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
        scores = gpkge.score_tails(h, r)
        target = scores[torch.arange(len(t), device=device), t]
        ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

ranks = torch.tensor(ranks, dtype=torch.float)
mrr = (1/ranks).mean().item()
hits1 = (ranks <= 1).float().mean().item()
hits3 = (ranks <= 3).float().mean().item()
hits10 = (ranks <= 10).float().mean().item()

# ECE
pos = test_data.triples[np.random.choice(len(test_data), 1000, replace=False)]
neg = np.array([[h, r, np.random.randint(test_data.num_entities)] for h,r,t in pos])
all_t = np.vstack([pos, neg])
labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

with torch.no_grad():
    h, r, t = [torch.tensor(all_t[:,j], device=device) for j in range(3)]
    conf = torch.sigmoid(gpkge.score_triple(h, r, t)).cpu().numpy()

ece, _ = expected_calibration_error(conf, labels)
brier = brier_score(conf, labels)

# AUROC
id_t = test_data.triples[np.random.choice(len(test_data), 1000, replace=False)]
ood_t = create_ood_dataset(train_data, test_data, "random", 1000)

def unc(triples):
    with torch.no_grad():
        h, r, t = [torch.tensor(triples[:,j], device=device) for j in range(3)]
        return gpkge.predict_with_uncertainty(h, r, t)['total'].cpu().numpy()

auroc = compute_auroc(unc(id_t), unc(ood_t))

results["GP-KGE (Ours)"] = {
    "mrr": mrr, "hits@1": hits1, "hits@3": hits3, "hits@10": hits10,
    "ece": ece, "brier": brier, "auroc": auroc
}

print(f"\nGP-KGE: MRR={mrr:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")

---
## Final Results
---

In [ ]:
# Final Results
print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"{'Model':<20} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'Brier':>8} {'AUROC':>8}")
print("-" * 70)

for name, r in results.items():
    print(f"{name:<20} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['brier']:>8.4f} {r['auroc']:>8.4f}")

# Key comparison
if "GGPN" in results and "GP-KGE (Ours)" in results:
    ggpn_ece = results["GGPN"]["ece"]
    gpkge_ece = results["GP-KGE (Ours)"]["ece"]
    improvement = (ggpn_ece - gpkge_ece) / ggpn_ece * 100 if ggpn_ece > 0 else 0

    print("\n" + "=" * 70)
    print("KEY FINDING: Calibration Comparison")
    print("=" * 70)
    print(f"  GGPN ECE:     {ggpn_ece:.4f}")
    print(f"  GP-KGE ECE:   {gpkge_ece:.4f}")
    print(f"  Improvement:  {improvement:.1f}%")

In [ ]:
# Save final results
with open('/content/final_results.json', 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in results.items()}, f, indent=2)

print("Results saved to /content/final_results.json")

# Download
try:
    from google.colab import files
    files.download('/content/final_results.json')
except:
    pass